# Quick Start: English to Spanish Steering Vector

This notebook demonstrates how to use the `steering-vectors` library to optimize a steering vector that causes a language model to generate Spanish text instead of English.

This is a direct adaptation of the original `llm-steering-opt/quickstart.ipynb` using the new modular API.

## Setup

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from dotenv import load_dotenv
import os

# Import from the new steering-vectors library
from steering_vectors import (
    SteeringOptimizer,
    VectorSteering,
    HuggingFaceBackend,
    TrainingDatapoint,
    OptimizationConfig,
)

load_dotenv()

In [ ]:
hf_token = os.getenv("HF_TOKEN")

model_name = "google/gemma-2-2b"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16, token=hf_token)

In [ ]:
# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Using device: {device}")

## Task Definition: English to Spanish Switching

We'll optimize a steering vector that causes the model to generate Spanish instead of English, given an English prompt.

In [ ]:
# Our training prompt - a recipe introduction
prompt = """Some of my fondest childhood memories are from my summer vacations back when I was little. Every now and then, after a long day of playing outside, I would come back home to be greeted with the delicious smell of my grandma's hazelnut cake wafting out of the kitchen. In this recipe, I'll teach you how to make that very cake, and create your own summer memories.

"""

In [ ]:
# Generate an unsteered completion to see what the model normally produces
generated = model.generate(
    **tokenizer(prompt, return_tensors="pt").to(device),
    max_new_tokens=15
)
generated_str = tokenizer.decode(generated[0], skip_special_tokens=True)
print("Unsteered completion:")
print(generated_str.replace(prompt, ""))

In [ ]:
# Define our completions: suppress English, promote Spanish
en_completion = """<h2>Ingredients</h2>

* 1 cup of all-purpose flour"""

es_completion = """<h2>Ingredientes</h2>

* 1 taza de harina común"""

## Optimizing the Steering Vector

Now we use the `steering-vectors` library's modular API:
1. Create a `HuggingFaceBackend` to handle model operations
2. Create a `VectorSteering` mode for additive steering
3. Configure with `OptimizationConfig`
4. Run optimization with `SteeringOptimizer`

In [ ]:
# Create our training datapoint
datapoint = TrainingDatapoint(
    prompt=prompt,
    src_completions=[en_completion],  # Suppress English
    dst_completions=[es_completion],  # Promote Spanish
)

In [ ]:
# Set up the components
backend = HuggingFaceBackend(model, tokenizer)
steering = VectorSteering()

# Configure optimization (matching original: lr=0.1, max_iters=20)
config = OptimizationConfig(
    lr=0.1,
    max_iters=20,
)

# Create optimizer
optimizer = SteeringOptimizer(backend, steering, config)

In [ ]:
# Run optimization at layer 10
layer = 10
result = optimizer.optimize([datapoint], layer=layer)

print(f"Optimization complete!")
print(f"  Iterations: {result.iterations}")
print(f"  Final loss: {result.final_loss:.4f}")
print(f"  Vector norm: {steering.get_vector().norm().item():.2f}")

## Testing the Steering Vector

Let's see if our optimized vector causes the model to generate Spanish!

In [ ]:
# Create the steering hook
hook = steering.create_hook()

# Generate with steering
with backend.hooks_context([(layer, hook)]):
    generated = model.generate(
        **tokenizer(prompt, return_tensors="pt").to(device),
        max_new_tokens=30
    )

steered_str = tokenizer.decode(generated[0], skip_special_tokens=True)
print("Steered completion:")
print(steered_str)

### Generalization Test

Does the vector generalize to other prompts?

In [ ]:
test_prompt = "My favorite programming languages are"
max_new_tokens = 35

print("--- Unsteered generation ---")
generated = model.generate(
    **tokenizer(test_prompt, return_tensors="pt").to(device),
    max_new_tokens=max_new_tokens
)
print(tokenizer.decode(generated[0], skip_special_tokens=True))
print()

print("--- Steered generation ---")
with backend.hooks_context([(layer, hook)]):
    generated = model.generate(
        **tokenizer(test_prompt, return_tensors="pt").to(device),
        max_new_tokens=max_new_tokens
    )
print(tokenizer.decode(generated[0], skip_special_tokens=True))

### Reverse Steering

If we negate the vector, we can switch from Spanish to English!

In [ ]:
spanish_prompt = "Unos de mis lenguajes de programación favoritos incluyen"
max_new_tokens = 30

print("--- Unsteered generation (Spanish prompt) ---")
generated = model.generate(
    **tokenizer(spanish_prompt, return_tensors="pt").to(device),
    max_new_tokens=max_new_tokens
)
print(tokenizer.decode(generated[0], skip_special_tokens=True))
print()

print("--- Reverse-steered generation (should switch to English) ---")
# Use strength=-1 to negate the vector
reverse_hook = steering.create_hook(strength=-1.0)
with backend.hooks_context([(layer, reverse_hook)]):
    generated = model.generate(
        **tokenizer(spanish_prompt, return_tensors="pt").to(device),
        max_new_tokens=max_new_tokens
    )
print(tokenizer.decode(generated[0], skip_special_tokens=True))

## Norm-Constrained Steering

We can limit the vector's norm to prevent overly strong steering effects.

In [ ]:
# Create a new steering mode and optimizer with norm constraint
steering_constrained = VectorSteering()
config_constrained = OptimizationConfig(
    lr=0.1,
    max_iters=20,
    max_norm=20.0,  # Limit vector norm to 20
)

optimizer_constrained = SteeringOptimizer(backend, steering_constrained, config_constrained)
result_constrained = optimizer_constrained.optimize([datapoint], layer=layer)

print(f"Norm-constrained optimization:")
print(f"  Final loss: {result_constrained.final_loss:.4f}")
print(f"  Vector norm: {steering_constrained.get_vector().norm().item():.2f}")

In [ ]:
# Test the norm-constrained vector
hook_constrained = steering_constrained.create_hook()

print("--- Steered with norm-constrained vector ---")
with backend.hooks_context([(layer, hook_constrained)]):
    generated = model.generate(
        **tokenizer(test_prompt, return_tensors="pt").to(device),
        max_new_tokens=35
    )
print(tokenizer.decode(generated[0], skip_special_tokens=True))

## Summary

This notebook demonstrated the new `steering-vectors` library API:

| Component | Purpose |
|-----------|--------|
| `HuggingFaceBackend` | Wraps model for tokenization, forward passes |
| `VectorSteering` | Defines the steering method (additive) |
| `OptimizationConfig` | Hyperparameters (lr, max_iters, max_norm) |
| `TrainingDatapoint` | Training data (prompt + completions to promote/suppress) |
| `SteeringOptimizer` | Runs the optimization loop |

The library also supports `ClampSteering` and `AffineSteering` for more advanced use cases.